# 05 — Data Preprocessing & RNN Sequence Generation
**Project:** Reinforcement Learning with RNN-Based Market State Representation for Algorithmic Trading  
**Step:** Chronological split → Scaler fit (train only) → Sliding window generation → Save tensors.

**Outputs:**
- `data/training/train/X_train.pt`, `y_train.pt`, `train_data.npz`, `metadata_train.csv`
- `data/training/validation/` (same pattern)
- `data/training/test/` (same pattern)
- `data/training/scalers/feature_scaler.joblib`, `scaler_metadata.json`

## 1. Imports & Configuration

In [ ]:
import os
import sys
import json
import logging
from typing import Dict, List, Tuple, Any, Optional

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

PROCESSED_DIR = os.path.join("data", "processed")
TRAINING_DIR  = os.path.join("data", "training")
TRAIN_DIR     = os.path.join(TRAINING_DIR, "train")
VAL_DIR       = os.path.join(TRAINING_DIR, "validation")
TEST_DIR      = os.path.join(TRAINING_DIR, "test")
SCALER_DIR    = os.path.join(TRAINING_DIR, "scalers")

STOCK_SYMBOLS = ["RELIANCE", "TCS", "INFY", "HDFCBANK", "ITC"]

WINDOW_LENGTH = 30
TRAIN_RATIO   = 0.70
VAL_RATIO     = 0.15

STATE_FEATURE_COLS: List[str] = [
    "Daily_Return", "Log_Return", "HL_Pct", "Rolling_Vol_20",
    "SMA_Ratio_10", "SMA_Ratio_20", "SMA_Ratio_50",
    "EMA_Ratio_12", "EMA_Ratio_26", "RSI_14_Norm",
    "MACD_Line_Norm", "MACD_Signal_Norm", "MACD_Hist_Norm", "Volume_Ratio_20",
]

TARGET_COL      = "Target_Return_Next"
DATE_COL        = "Date"
TARGET_DATE_COL = "Target_Date_Next"

for d in [TRAIN_DIR, VAL_DIR, TEST_DIR, SCALER_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Window length  : {WINDOW_LENGTH} trading days")
print(f"Split          : {TRAIN_RATIO*100:.0f}% / {VAL_RATIO*100:.0f}% / {(1-TRAIN_RATIO-VAL_RATIO)*100:.0f}%")
print(f"State features : {len(STATE_FEATURE_COLS)}")

## 2. Chronological Split Function

In [ ]:
def chronological_split(
    df: pd.DataFrame,
    train_ratio: float = TRAIN_RATIO,
    val_ratio: float   = VAL_RATIO,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Splits a single asset's time series chronologically into train, val, and test subsets.

    Returns:
        (train_df, val_df, test_df) — strictly non-overlapping, past-to-future order.
    """
    n       = len(df)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)

    train_df = df.iloc[:n_train].copy().reset_index(drop=True)
    val_df   = df.iloc[n_train : n_train + n_val].copy().reset_index(drop=True)
    test_df  = df.iloc[n_train + n_val :].copy().reset_index(drop=True)
    return train_df, val_df, test_df

## 3. Load Data & Fit Scaler

In [ ]:
print("=" * 60)
print("STEP 4: Preprocessing & RNN Sequence Generation")
print("=" * 60)

raw_splits: Dict[str, Dict[str, pd.DataFrame]] = {}
all_train_features: List[np.ndarray] = []

for sym in STOCK_SYMBOLS:
    path = os.path.join(PROCESSED_DIR, f"{sym}_processed.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Processed file missing: {path}")

    df = pd.read_csv(path)
    train_df, val_df, test_df = chronological_split(df)
    raw_splits[sym] = {"train": train_df, "val": val_df, "test": test_df}
    all_train_features.append(train_df[STATE_FEATURE_COLS].values)

    logger.info(
        f"[{sym}] Train={len(train_df)} "
        f"[{train_df[DATE_COL].iloc[0]} → {train_df[DATE_COL].iloc[-1]}] | "
        f"Val={len(val_df)} | Test={len(test_df)}"
    )

# Fit StandardScaler EXCLUSIVELY on training partition
train_matrix = np.vstack(all_train_features)
scaler = StandardScaler()
scaler.fit(train_matrix)
logger.info(f"Fitted StandardScaler on {train_matrix.shape[0]} training rows × {train_matrix.shape[1]} features")

# Save scaler artifacts
scaler_path = os.path.join(SCALER_DIR, "feature_scaler.joblib")
joblib.dump(scaler, scaler_path)
logger.info(f"Saved scaler → {scaler_path}")

meta_dict = {
    "feature_names": STATE_FEATURE_COLS,
    "mean":          scaler.mean_.tolist(),
    "scale":         scaler.scale_.tolist(),
    "var":           scaler.var_.tolist(),
    "n_samples_seen": int(scaler.n_samples_seen_),
}
meta_path = os.path.join(SCALER_DIR, "scaler_metadata.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta_dict, f, indent=4)
logger.info(f"Saved scaler metadata → {meta_path}")

## 4. Sliding Window Generation

In [ ]:
aggregated: Dict[str, Dict[str, Any]] = {
    "train":      {"X": [], "y": [], "meta": []},
    "validation": {"X": [], "y": [], "meta": []},
    "test":       {"X": [], "y": [], "meta": []},
}

for sym in STOCK_SYMBOLS:
    train_df = raw_splits[sym]["train"]
    val_df   = raw_splits[sym]["val"]
    test_df  = raw_splits[sym]["test"]

    train_scaled = scaler.transform(train_df[STATE_FEATURE_COLS].values)
    val_scaled   = scaler.transform(val_df[STATE_FEATURE_COLS].values)
    test_scaled  = scaler.transform(test_df[STATE_FEATURE_COLS].values)

    # ── Train windows ─────────────────────────────────────────────
    T, D = train_scaled.shape
    n_tr = T - WINDOW_LENGTH + 1
    X_tr = np.empty((n_tr, WINDOW_LENGTH, D), dtype=np.float32)
    y_tr = np.empty((n_tr, 1), dtype=np.float32)
    meta_tr = []
    for i in range(n_tr):
        end = i + WINDOW_LENGTH - 1
        X_tr[i]  = train_scaled[i : i + WINDOW_LENGTH]
        y_tr[i, 0] = train_df[TARGET_COL].values[end]
        meta_tr.append({
            "Sample_Index":    i,
            "Symbol":          sym,
            "Window_Start_Date": train_df[DATE_COL].iloc[i],
            "Window_End_Date":   train_df[DATE_COL].iloc[end],
            "Target_Date":       train_df[TARGET_DATE_COL].iloc[end],
            "Target_Return":   float(train_df[TARGET_COL].values[end]),
        })

    # ── Val windows (prepend last L-1 train rows as context) ──────
    ctx_v = train_scaled[-(WINDOW_LENGTH - 1):]
    val_ext = np.vstack([ctx_v, val_scaled])
    val_dates_ext = (
        train_df[DATE_COL].iloc[-(WINDOW_LENGTH - 1):].tolist() +
        val_df[DATE_COL].tolist()
    )
    n_va = len(val_df)
    X_va = np.empty((n_va, WINDOW_LENGTH, D), dtype=np.float32)
    y_va = np.empty((n_va, 1), dtype=np.float32)
    meta_va = []
    for i in range(n_va):
        X_va[i]  = val_ext[i : i + WINDOW_LENGTH]
        y_va[i, 0] = val_df[TARGET_COL].values[i]
        meta_va.append({
            "Sample_Index":    i,
            "Symbol":          sym,
            "Window_Start_Date": val_dates_ext[i],
            "Window_End_Date":   val_df[DATE_COL].iloc[i],
            "Target_Date":       val_df[TARGET_DATE_COL].iloc[i],
            "Target_Return":   float(val_df[TARGET_COL].values[i]),
        })

    # ── Test windows (prepend last L-1 val rows as context) ───────
    ctx_t = val_scaled[-(WINDOW_LENGTH - 1):]
    test_ext = np.vstack([ctx_t, test_scaled])
    test_dates_ext = (
        val_df[DATE_COL].iloc[-(WINDOW_LENGTH - 1):].tolist() +
        test_df[DATE_COL].tolist()
    )
    n_te = len(test_df)
    X_te = np.empty((n_te, WINDOW_LENGTH, D), dtype=np.float32)
    y_te = np.empty((n_te, 1), dtype=np.float32)
    meta_te = []
    for i in range(n_te):
        X_te[i]  = test_ext[i : i + WINDOW_LENGTH]
        y_te[i, 0] = test_df[TARGET_COL].values[i]
        meta_te.append({
            "Sample_Index":    i,
            "Symbol":          sym,
            "Window_Start_Date": test_dates_ext[i],
            "Window_End_Date":   test_df[DATE_COL].iloc[i],
            "Target_Date":       test_df[TARGET_DATE_COL].iloc[i],
            "Target_Return":   float(test_df[TARGET_COL].values[i]),
        })

    # Save per-stock tensors
    torch.save({"X": torch.from_numpy(X_tr), "y": torch.from_numpy(y_tr)}, os.path.join(TRAIN_DIR, f"{sym}_train.pt"))
    torch.save({"X": torch.from_numpy(X_va), "y": torch.from_numpy(y_va)}, os.path.join(VAL_DIR,   f"{sym}_val.pt"))
    torch.save({"X": torch.from_numpy(X_te), "y": torch.from_numpy(y_te)}, os.path.join(TEST_DIR,  f"{sym}_test.pt"))

    aggregated["train"]["X"].append(X_tr);       aggregated["train"]["y"].append(y_tr);       aggregated["train"]["meta"].extend(meta_tr)
    aggregated["validation"]["X"].append(X_va);  aggregated["validation"]["y"].append(y_va);  aggregated["validation"]["meta"].extend(meta_va)
    aggregated["test"]["X"].append(X_te);        aggregated["test"]["y"].append(y_te);        aggregated["test"]["meta"].extend(meta_te)

print("Sliding windows generated for all stocks.")

## 5. Save Combined Multi-Stock Datasets

In [ ]:
split_dirs  = {"train": TRAIN_DIR, "validation": VAL_DIR, "test": TEST_DIR}
split_keys  = {"train": "train",   "validation": "val",   "test": "test"}

for split_label, s_key in split_keys.items():
    dir_path   = split_dirs[split_label]
    combined_X = np.concatenate(aggregated[split_label]["X"], axis=0)
    combined_y = np.concatenate(aggregated[split_label]["y"], axis=0)
    meta_df    = pd.DataFrame(aggregated[split_label]["meta"])

    # PyTorch tensors
    torch.save(torch.from_numpy(combined_X), os.path.join(dir_path, f"X_{s_key}.pt"))
    torch.save(torch.from_numpy(combined_y), os.path.join(dir_path, f"y_{s_key}.pt"))

    # NumPy compressed
    np.savez_compressed(
        os.path.join(dir_path, f"{s_key}_data.npz"),
        X=combined_X, y=combined_y,
        symbols=meta_df["Symbol"].values,
        target_dates=meta_df["Target_Date"].values,
    )

    # Metadata CSV
    meta_df.to_csv(os.path.join(dir_path, f"metadata_{s_key}.csv"), index=False)

    logger.info(f"Saved {split_label.upper()}: X={combined_X.shape}, y={combined_y.shape} → {dir_path}")

print("\nPreprocessing complete — all training artifacts saved.")

## 6. Summary Table

In [ ]:
print("\nFinal Dataset Shapes:")
print(f"{'Split':<12} {'X shape':>20}  {'y shape':>12}  {'Min Target Date':>15}  {'Max Target Date':>15}")
print("-" * 85)
for sl, sk in split_keys.items():
    X = torch.load(os.path.join(split_dirs[sl], f"X_{sk}.pt"), weights_only=True)
    y = torch.load(os.path.join(split_dirs[sl], f"y_{sk}.pt"), weights_only=True)
    meta = pd.read_csv(os.path.join(split_dirs[sl], f"metadata_{sk}.csv"))
    print(
        f"{sl:<12} {str(tuple(X.shape)):>20}  {str(tuple(y.shape)):>12}  "
        f"{meta['Target_Date'].min():>15}  {meta['Target_Date'].max():>15}"
    )